In [68]:
import os
from dotenv import load_dotenv
load_dotenv()
# 환경 변수 읽기
OPENDART_API_KEY = os.getenv("OPENDART_API_KEY")

# 기업별고유번호

In [65]:
import requests
import zipfile
import io
import xml.etree.ElementTree as ET
import pandas as pd


url = "https://opendart.fss.or.kr/api/corpCode.xml"
params = {"crtfc_key": OPENDART_API_KEY}

# 1. API 호출 (ZIP 파일 수신)
response = requests.get(url, params=params)
response.raise_for_status()

# 2. ZIP 파일 열기 (메모리에서 바로 처리)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # 보통 하나의 XML 파일만 존재
    xml_filename = z.namelist()[0]
    
    with z.open(xml_filename) as xml_file:
        xml_data = xml_file.read()

# 3. XML 파싱
root = ET.fromstring(xml_data)

rows = []
for corp in root.findall(".//list"):
    rows.append({
        "corp_code": corp.findtext("corp_code"),
        "corp_name": corp.findtext("corp_name"),
        "stock_code": corp.findtext("stock_code"),
        "modify_date": corp.findtext("modify_date"),
    })

# 4. DataFrame 변환
df = pd.DataFrame(rows)
df.to_csv("../data/기업별고유번호.csv", encoding="utf-8-sig", index=False)

print(df.head())
print(df.shape)

  corp_code          corp_name stock_code modify_date
0  00434003                 다코               20170630
1  00430964              굿앤엘에스               20170630
2  00388953  크레디피아제이십오차유동화전문회사               20170630
3  00179984             연방건설산업               20170630
4  00420143     브룩스피알아이오토메이션잉크               20170630
(117496, 4)


In [67]:
corps = pd.read_csv("../data/기업별고유번호.csv", dtype={'corp_code':str,'stock_code':str})
# 기업 고유번호가 0으로 시작하는 경우 int로 읽어오면 앞자리 0이 사라지기 때문에
# dtype(데이터 타입)을 str로 지정해줍니다.
corps = corps[corps['stock_code'].notna()].copy()
corps

,corp_code,corp_name,stock_code,modify_date
0,00434003,다코,,20170630
1,00430964,굿앤엘에스,,20170630
2,00388953,크레디피아제이십오차유동화전문회사,,20170630
3,00179984,연방건설산업,,20170630
4,00420143,브룩스피알아이오토메이션잉크,,20170630
...,...,...,...,...
117491,00652043,한솔오리온텍,,20250918
117492,01956955,도경회계법인,,20250918
117493,01956964,새중앙새마을금고,,20250918
117494,01956973,현대인베스트선진인더스트리얼일반사모부동산투자회사,,20250918


In [73]:
df_filtered = corps[corps["corp_name"].str.contains("현대중공업", case=False, na=False)]
df_filtered

,corp_code,corp_name,stock_code,modify_date
12568,00887676,코에프씨현대중공업협력사동반성장제일호사모투자전문회사,,20170630
53310,01610549,현대중공업그룹일퍼센트나눔재단,,20220119
71725,01336179,현대중공업파워시스템,,20230215
88558,01169346,에이치디현대중공업모스,,20230407
109183,01135349,현대중공업터보기계,,20251209
111507,01390344,HD현대중공업,329180,20260401


# OPENDART API

In [7]:
import requests
def get_dart_data(url, params):
    return requests.get(url, params=params)

# 공시정보 및 정기보고서 주요정보

In [22]:
공시검색 = {
    "url": "https://opendart.fss.or.kr/api/list.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de}
}
기업개황 = {
    "url": "https://opendart.fss.or.kr/api/company.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code}
}
증감자현황 = {
    "url": "https://opendart.fss.or.kr/api/irdsSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
배당현황 = {
    "url": "https://opendart.fss.or.kr/api/alotMatter.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
자기주식취득현황 = {
    "url": "https://opendart.fss.or.kr/api/tesstkAcqsDspsSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
최대주주현황 = {
    "url": "https://opendart.fss.or.kr/api/hyslrSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
최대주주변동현황 = {
    "url": "https://opendart.fss.or.kr/api/hyslrChgSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
임원현황 = {
    "url": "https://opendart.fss.or.kr/api/exctvSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
직원현황 = {
    "url": "https://opendart.fss.or.kr/api/empSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
이사감사개인별보수현황 = {
    "url": "https://opendart.fss.or.kr/api/hmvAuditIndvdlBySttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
이사감사전체보수현황 = {
    "url": "https://opendart.fss.or.kr/api/hmvAuditAllSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
개인별보수현황_top5 = {
    "url": "https://opendart.fss.or.kr/api/indvdlByPay.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
타법인출자현황 = {
    "url": "https://opendart.fss.or.kr/api/otrCprInvstmntSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
주식총수현황 = {
    "url": "https://opendart.fss.or.kr/api/stockTotqySttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
채무증권발행실적 = {
    "url": "https://opendart.fss.or.kr/api/detScritsIsuAcmslt.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
어음미상환잔액 = {
    "url": "https://opendart.fss.or.kr/api/entrprsBilScritsNrdmpBlce.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
단기사채미상환잔액 = {
    "url": "https://opendart.fss.or.kr/api/srtpdPsndbtNrdmpBlce.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
회사채미상환잔액 = {
    "url": "https://opendart.fss.or.kr/api/cprndNrdmpBlce.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
신종자본미상환잔액 = {
    "url": "https://opendart.fss.or.kr/api/newCaplScritsNrdmpBlce.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
조건부자본미상환잔액 = {
    "url": "https://opendart.fss.or.kr/api/cndlCaplScritsNrdmpBlce.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
감사의견 = {
    "url": "https://opendart.fss.or.kr/api/accnutAdtorNmNdAdtOpinion.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
감사용역체결현황 = {
    "url": "https://opendart.fss.or.kr/api/adtServcCnclsSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
비감사용역체결현황 = {
    "url": "https://opendart.fss.or.kr/api/accnutAdtorNonAdtServcCnclsSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
사외이사현황 = {
    "url": "https://opendart.fss.or.kr/api/outcmpnyDrctrNdChangeSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
미등기임원보수현황 = {
    "url": "https://opendart.fss.or.kr/api/unrstExctvMendngSttus.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
이사감사전체보수현황_주총승인 = {
    "url": "https://opendart.fss.or.kr/api/drctrAdtAllMendngSttusGmtsckConfmAmount.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
이사감사전체보수현황_유형별 = {
    "url": "https://opendart.fss.or.kr/api/drctrAdtAllMendngSttusMendngPymntamtTyCl.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
공모자금사용내역 = {
    "url": "https://opendart.fss.or.kr/api/pssrpCptalUseDtls.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
사모자금사용내역 = {
    "url": "https://opendart.fss.or.kr/api/prvsrpCptalUseDtls.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}

In [26]:
target = 미등기임원보수현황
res = get_dart_data(url=target["url"], params=target["params"])
results = res.json()
results

{'status': '000',
 'message': '정상',
 'list': [{'rcept_no': '20260310002820',
   'corp_cls': 'Y',
   'corp_code': '00126380',
   'corp_name': '삼성전자',
   'se': '미등기임원',
   'fyer_salary_totamt': '705,452,000,000',
   'jan_salary_am': '744,000,000',
   'nmpr': '989',
   'rm': '-',
   'stlm_dt': '2025-12-31'}]}

In [27]:
import pandas as pd
df = pd.DataFrame(results['list'])
print(df.shape)
df.head(10)

(1, 10)


,rcept_no,corp_cls,corp_code,corp_name,se,fyer_salary_totamt,jan_salary_am,nmpr,rm,stlm_dt
0,20260310002820,Y,00126380,삼성전자,미등기임원,"705,452,000,000","744,000,000",989,-,2025-12-31


# 정기보고서 재무정보

In [ ]:
단일회사주요계정 = {
    "url": "https://opendart.fss.or.kr/api/fnlttSinglAcnt.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
다중회사주요계정 = {
    "url": "https://opendart.fss.or.kr/api/fnlttMultiAcnt.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code}
}
단일회사전체재무재표 = {
    "url": "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code, "fs_div": fs_div}
}
단일회사주요재무지표 = {
    "url": "https://opendart.fss.or.kr/api/fnlttSinglIndx.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code, "idx_cl_code": idx_cl_code}
}
다중회사주요재무지표 = {
    "url": "https://opendart.fss.or.kr/api/prvsrpCptalUseDtls.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code":reprt_code, "idx_cl_code": idx_cl_code}
}

In [33]:
target = 단일회사주요재무지표
res = get_dart_data(url=target["url"], params=target["params"])
results = res.json()
results

{'status': '000',
 'message': '정상',
 'list': [{'reprt_code': '11011',
   'bsns_year': '2025',
   'corp_code': '00126380',
   'stock_code': '005930',
   'stlm_dt': '2025-12-31',
   'idx_cl_code': 'M210000',
   'idx_cl_nm': '수익성지표',
   'idx_code': 'M211100',
   'idx_nm': '세전계속사업이익률'},
  {'reprt_code': '11011',
   'bsns_year': '2025',
   'corp_code': '00126380',
   'stock_code': '005930',
   'stlm_dt': '2025-12-31',
   'idx_cl_code': 'M210000',
   'idx_cl_nm': '수익성지표',
   'idx_code': 'M211200',
   'idx_nm': '순이익률',
   'idx_val': '13.551'},
  {'reprt_code': '11011',
   'bsns_year': '2025',
   'corp_code': '00126380',
   'stock_code': '005930',
   'stlm_dt': '2025-12-31',
   'idx_cl_code': 'M210000',
   'idx_cl_nm': '수익성지표',
   'idx_code': 'M211250',
   'idx_nm': '총포괄이익률',
   'idx_val': '15.375'},
  {'reprt_code': '11011',
   'bsns_year': '2025',
   'corp_code': '00126380',
   'stock_code': '005930',
   'stlm_dt': '2025-12-31',
   'idx_cl_code': 'M210000',
   'idx_cl_nm': '수익성지표',
   'idx_c

In [34]:
import pandas as pd
df = pd.DataFrame(results['list'])
print(df.shape)
df.head(10)

(15, 10)


,reprt_code,bsns_year,corp_code,stock_code,stlm_dt,idx_cl_code,idx_cl_nm,idx_code,idx_nm,idx_val
0,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211100,세전계속사업이익률,NaN
1,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211200,순이익률,13.551
2,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211250,총포괄이익률,15.375
3,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211300,매출총이익률,39.379
4,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211400,매출원가율,60.621
5,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211550,ROE,10.783
6,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M211800,판관비율,26.309
7,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M212000,총자산영업이익률,8.063
8,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M212100,총자산세전계속사업이익률,NaN
9,11011,2025,00126380,005930,2025-12-31,M210000,수익성지표,M212200,자기자본영업이익률,10.4


# 지분공시 종합정보

In [78]:
대량보유상황보고 = {
    "url": "https://opendart.fss.or.kr/api/majorstock.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code}
}
임원주요주주소유보고 = {
    "url": " 	https://opendart.fss.or.kr/api/elestock.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code}
}

# 주요사항보고서 주요 정보

In [42]:
자산양수도풋백옵션 = {
    "url": "https://opendart.fss.or.kr/api/astInhtrfEtcPtbkOpt.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
부도발생 = {
    "url": "https://opendart.fss.or.kr/api/dfOcr.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
영업정지 = {
    "url": "https://opendart.fss.or.kr/api/bsnSp.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
회생절차개시신청 = {
    "url": "https://opendart.fss.or.kr/api/ctrcvsBgrq.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
해산사유발생 = {
    "url": "https://opendart.fss.or.kr/api/dsRsOcr.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
유상증자결정 = {
    "url": "https://opendart.fss.or.kr/api/piicDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
무상증자결정 = {
    "url": "https://opendart.fss.or.kr/api/fricDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
유무상증자결정 = {
    "url": "https://opendart.fss.or.kr/api/pifricDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
감자결정 = {
    "url": "https://opendart.fss.or.kr/api/crDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
채권은행관리개시 = {
    "url": "https://opendart.fss.or.kr/api/bnkMngtPcbg.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
소송등의제기 = {
    "url": "https://opendart.fss.or.kr/api/lwstLg.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
해외증권시장주권상장결정 = {
    "url": "https://opendart.fss.or.kr/api/ovLstDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
해외증권시장주권상장폐지결정 = {
    "url": "https://opendart.fss.or.kr/api/ovDlstDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
해외증권시장주권등상장 = {
    "url": "https://opendart.fss.or.kr/api/ovLst.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
해외증권시장주권등상장폐지 = {
    "url": "https://opendart.fss.or.kr/api/ovDlst.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
전환사채발행결정 = {
    "url": "https://opendart.fss.or.kr/api/cvbdIsDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
신주인수권부사채발행결정 = {
    "url": "https://opendart.fss.or.kr/api/bdwtIsDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
교환사채발행결정 = {
    "url": "https://opendart.fss.or.kr/api/exbdIsDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
채권은행의관리절차중단 = {
    "url": "https://opendart.fss.or.kr/api/bnkMngtPcsp.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
상각형조건부자본증권발행결정 = {
    "url": "https://opendart.fss.or.kr/api/wdCocobdIsDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
자기주식취득결정 = {
    "url": "https://opendart.fss.or.kr/api/tsstkAqDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
자기주식취득신탁계약해지결정 = {
    "url": "https://opendart.fss.or.kr/api/tsstkAqTrctrCcDecsn.jsonn",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
영업양수결정 = {
    "url": "https://opendart.fss.or.kr/api/bsnInhDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
영업양도결정 = {
    "url": "https://opendart.fss.or.kr/api/bsnTrfDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
유형자산양수결정 = {
    "url": "https://opendart.fss.or.kr/api/tgastInhDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
유형자산양도결정 = {
    "url": "https://opendart.fss.or.kr/api/tgastTrfDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
타법인주식출자증권양수결정 = {
    "url": "https://opendart.fss.or.kr/api/otcprStkInvscrInhDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
타법인주식출자증권양도결정 = {
    "url": "https://opendart.fss.or.kr/api/otcprStkInvscrTrfDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
주권관련사채권양수결정 = {
    "url": "https://opendart.fss.or.kr/api/stkrtbdInhDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
주권관련사채권양도결정 = {
    "url": "https://opendart.fss.or.kr/api/stkrtbdTrfDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
회사합병결정 = {
    "url": "https://opendart.fss.or.kr/api/cmpMgDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
회사분할결정 = {
    "url": "https://opendart.fss.or.kr/api/cmpDvDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
회사분할합병결정 = {
    "url": "https://opendart.fss.or.kr/api/cmpDvmgDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
주식교환이전결정 = {
    "url": "https://opendart.fss.or.kr/api/stkExtrDecsn.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}

# 증권신고서 주요정보

In [50]:
지분증권 = {
    "url": "https://opendart.fss.or.kr/api/estkRs.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
채무증권 = {
    "url": "https://opendart.fss.or.kr/api/bdRs.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
증권예탁증권 = {
    "url": "https://opendart.fss.or.kr/api/stkdpRs.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
합병 = {
    "url": "https://opendart.fss.or.kr/api/mgRs.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
주식포괄적교환이전 = {
    "url": "https://opendart.fss.or.kr/api/extrRs.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}
분할 = {
    "url": "https://opendart.fss.or.kr/api/dvRs.json",
    "params": {"crtfc_key": OPENDART_API_KEY, "corp_code": corp_code, "bgn_de": bgn_de, "end_de": end_de}
}

# 테스트

In [76]:
corp_code = "01390344"  
bgn_de = "20260101"  # 시작일(최초접수일)
end_de = "20260430"  # 종료일(최초접수일)
last_reprt_at = "Y"  # 최종보고서 검색여부
bsns_year = "2025"   # 사업연도
reprt_code = "11011" # 보고서코드 (1분기보고서 : 11013, 반기보고서 : 11012, 3분기보고서 : 11014, 사업보고서 : 11011)
fs_div = "OFS"       # 개별/연결구분 (OFS:재무제표, CFS:연결재무제표)
idx_cl_code = "M210000"  # 지표분류코드 (수익성지표 : M210000 안정성지표 : M220000 성장성지표 : M230000 활동성지표 : M240000)

In [81]:
target = 대량보유상황보고
res = get_dart_data(url=target["url"], params=target["params"])
results = res.json()
import pandas as pd
df = pd.DataFrame(results['list'])
print(df.shape)
df.head(50)

(5, 13)


,rcept_no,rcept_dt,corp_code,corp_name,report_tp,repror,stkqy,stkqy_irds,stkrt,stkrt_irds,ctr_stkqy,ctr_stkrt,report_resn
0,20240521000292,2024-05-21,01390344,HD현대중공업,일반,HD한국조선해양,"66,602,223","-2,663,000",75.02,-3.00,-,-,본인의 보유주식 일부 단순처분
1,20250228001875,2025-02-28,01390344,HD현대중공업,일반,HD한국조선해양,"66,602,223",0,75.02,0.00,"1,730,576",1.95,보고자의 보유주식에 관한 계약 체결(교환사채권 인수계약)
2,20250702000080,2025-07-02,01390344,HD현대중공업,약식,국민연금공단,"6,554,502","888,755",7.38,1.00,-,-,단순추가취득/처분
3,20251208000185,2025-12-08,01390344,HD현대중공업,일반,HD한국조선해양,"72,739,290","6,137,067",69.30,-5.72,"1,730,576",1.95,- 교환사채권 행사에 따른 주식 수 변동\n- 합병에 따른 주식 수 변동\n- 합병...
4,20260401004888,2026-04-01,01390344,HD현대중공업,일반,HD한국조선해양,"72,712,576","-26,714",69.28,-0.02,"6,265,390",5.96,- 교환사채권 행사에 따른 주식 수 변동\n- 특별관계자 추가\n- 보고자의 보유주...
